In [ ]:
"""
Kafka Exporter를 활용하여 특정 컨슈머 그룹의 Lag을 모니터링하는 스크립트입니다.

동작 방식:
1. Prometheus API를 이용하여 Kafka Exporter의 /metrics 엔드포인트를 호출합니다.
2. 특정 컨슈머 그룹의 kafka_consumergroup_lag 값을 찾습니다.
3. 모든 토픽/파티션의 Lag 값을 합산합니다.
4. 일정 주기마다 Lag을 출력합니다.
5. Lag이 임계값 이상이면 경고 메시지를 출력합니다.
"""


In [1]:
import time
import requests

# Kafka Exporter metrics endpoint
EXPORTER_METRICS_URL = "http://localhost:9308/metrics"

# 모니터링할 컨슈머 그룹명
CONSUMER_GROUP = "test-group"

# Lag이 이 값 이상이면 경고 출력
LAG_THRESHOLD = 100

# Lag 확인 주기(초)
CHECK_INTERVAL = 5

In [2]:
# 1: Prometheus API를 호출하여 특정 컨슈머 그룹의 모든 파티션 Lag 합계를 반환하는 함수를 구현
def get_consumer_lag():
    response = requests.get(EXPORTER_METRICS_URL)
    lines = response.text.split('\n')
    total_lag = 0

    for line in lines:
        if line.startswith("kafka_consumergroup_lag") and f'consumergroup="{CONSUMER_GROUP}"' in line:
            lag_value = float(line.split()[-1])
            total_lag += lag_value

    return total_lag

In [7]:
while True:
    lag = get_consumer_lag()
    print(f"Current Lag for {CONSUMER_GROUP}: {lag}")

    # 2: Lag이 임계값 이상이면 경고 메시지를 출력
    if lag >= LAG_THRESHOLD:
        print("WARNING: Consumer Lag is too high!")

    time.sleep(CHECK_INTERVAL)  # 3: 일정 주기마다 Lag을 체크

Current Lag for test-group: 732.0
Current Lag for test-group: 10756.0
Current Lag for test-group: 20842.0
Current Lag for test-group: 30978.0
Current Lag for test-group: 41000.0
Current Lag for test-group: 50992.0
Current Lag for test-group: 59452.0
Current Lag for test-group: 68000.0
Current Lag for test-group: 76454.0
Current Lag for test-group: 2768.0
Current Lag for test-group: 2846.0
Current Lag for test-group: 2858.0
Current Lag for test-group: 11650.0
Current Lag for test-group: 7100.0


KeyboardInterrupt: 